# Graham scan: watch the hull take shape

A visual companion to the [Graham scan lesson](../algorithms/geometry/graham-scan.md). Move through the sorted points one at a time and watch the stack change. The notebook uses a small planar example; it is not geographic data.

Open in [Google Colab](https://colab.research.google.com/github/geo-dev-hub/spatial-analysis/blob/main/notebooks/graham-scan-lab.ipynb) or run in JupyterLab with `notebooks/requirements.txt`.

**Learning goal:** connect polar-angle sorting, the orientation test and stack updates to the final convex boundary.

In [ ]:
from pathlib import Path
import sys
from math import atan2

import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact

current = Path.cwd().resolve()
repo_root = next((p for p in (current, *current.parents)
                  if (p / 'implementations/python/graham-scan').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
sys.path.insert(0, str(repo_root / 'implementations/python/graham-scan'))
from graham_scan import cross, graham_scan

print('Repository:', repo_root)

## 1. Start with a set of points

The outer boundary is not known yet. First we choose the anchor: the point with the smallest y-coordinate, breaking ties with the smallest x-coordinate. Then we sort the remaining points by their polar angle around that anchor.

Points on the same ray are ordered from nearest to farthest. This matters because the farther point is the extreme one.

In [ ]:
points = [
    (0, 0), (2, 0), (3, 1), (2, 2), (0, 2),
    (1, 1), (1, 0.7), (0.8, 1.5), (2.2, 1.2), (1.5, 1.7),
]

def polar_order(points):
    unique = sorted(set((float(x), float(y)) for x, y in points))
    if len(unique) <= 1:
        return unique[0] if unique else None, []
    anchor = min(unique, key=lambda p: (p[1], p[0]))
    others = [p for p in unique if p != anchor]
    others.sort(key=lambda p: (
        atan2(p[1] - anchor[1], p[0] - anchor[0]),
        (p[0] - anchor[0]) ** 2 + (p[1] - anchor[1]) ** 2,
    ))
    return anchor, others

anchor, ordered = polar_order(points)
print('Anchor:', anchor)
for index, point in enumerate(ordered, start=1):
    print(f'{index:>2}. {point}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(*zip(*points), s=55, label='Input points')
ax.scatter([anchor[0]], [anchor[1]], s=150, marker='*', label='Anchor')
for index, point in enumerate(ordered, start=1):
    ax.annotate(str(index), point, xytext=(5, 5), textcoords='offset points')
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Points in polar-angle order around the anchor')
ax.grid(True)
ax.legend()
plt.show()

## 2. Watch the stack update

For each new point, inspect the last two points already on the stack. If they and the candidate do not make a counter-clockwise turn, remove the last stack point. Repeat until the turn is counter-clockwise, then add the candidate.

The slider shows the stack after each candidate has been processed. Removed points are marked separately so you can see why the boundary changes.

In [ ]:
def scan_trace(points):
    unique = sorted(set((float(x), float(y)) for x, y in points))
    if len(unique) <= 2:
        return (unique[0] if unique else None), [], [unique.copy()], [[]]
    anchor = min(unique, key=lambda p: (p[1], p[0]))
    ordered = [p for p in unique if p != anchor]
    ordered.sort(key=lambda p: (
        atan2(p[1] - anchor[1], p[0] - anchor[0]),
        (p[0] - anchor[0]) ** 2 + (p[1] - anchor[1]) ** 2,
    ))
    stack = [anchor]
    states = [stack.copy()]
    removed_by_step = [[]]
    for point in ordered:
        removed = []
        while len(stack) >= 2 and cross(stack[-2], stack[-1], point) <= 0:
            removed.append(stack.pop())
        stack.append(point)
        states.append(stack.copy())
        removed_by_step.append(removed)
    return anchor, ordered, states, removed_by_step

anchor, ordered, states, removed_by_step = scan_trace(points)
final_hull = graham_scan(points)
assert states[-1] == final_hull

def show_step(step=0):
    stack = states[step]
    removed = removed_by_step[step]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    ax.scatter(*zip(*points), s=45, label='All input points')
    ax.scatter([anchor[0]], [anchor[1]], s=140, marker='*', label='Anchor')
    if stack:
        ax.plot([p[0] for p in stack], [p[1] for p in stack], marker='o', linewidth=2.5, label='Current stack')
    if removed:
        ax.scatter([p[0] for p in removed], [p[1] for p in removed], s=120, marker='x', label='Popped this step')
    ax.set_title(f'Stack after {step} of {len(ordered)} candidates')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box'); ax.grid(True); ax.legend()

    ax = axes[1]
    ax.scatter(*zip(*points), s=45, label='Input points')
    closed = final_hull + [final_hull[0]] if len(final_hull) > 1 else final_hull
    if len(closed) > 1:
        ax.plot([p[0] for p in closed], [p[1] for p in closed], marker='o', linewidth=2.5, label='Final convex hull')
    ax.set_title('Final hull for comparison')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box'); ax.grid(True); ax.legend()
    plt.show()
    if step == 0:
        print('Only the anchor is on the stack. The first sorted point will be added next.')
    else:
        candidate = ordered[step - 1]
        print('Candidate:', candidate)
        print('Points removed:', removed)
        if len(stack) >= 2:
            print('Last turn after insertion:', cross(stack[-2], stack[-1], candidate) if stack[-1] != candidate else 'candidate is now the stack top')
        print('Current stack:', stack)

interact(show_step, step=IntSlider(value=0, min=0, max=len(ordered), step=1, description='Candidate'))

## 3. Check the result

The final stack is the hull in counter-clockwise order. The first point is not repeated at the end. This teaching implementation removes collinear middle points, so only extreme vertices remain.

Run the implementation independently and compare it with the trace.

In [ ]:
result = graham_scan(points)
print('Hull vertices:')
for point in result:
    print(point)
print('Matches final trace:', result == states[-1])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(*zip(*points), s=55, label='Input points')
closed = result + [result[0]] if len(result) > 1 else result
if len(closed) > 1:
    ax.plot([p[0] for p in closed], [p[1] for p in closed], marker='o', linewidth=3, label='Convex hull')
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('The extreme boundary')
ax.grid(True); ax.legend()
plt.show()

## 4. Experiments

1. Add a point well inside the hull. Does the result change?
2. Add a point farther out along an existing hull edge. Which old vertex disappears?
3. Make every point collinear. The hull should collapse to the two extreme endpoints.
4. Duplicate one coordinate. The hull should remain unchanged.
5. Try nearly collinear floating-point coordinates. Why might a production implementation need a robust orientation predicate?

## Continue learning

- [Graham scan lesson](../algorithms/geometry/graham-scan.md)
- [Python implementation](../implementations/python/graham-scan/README.md)
- [Step-by-step implementation walkthrough](../implementations/python/graham-scan/implementation-walkthrough.md)
- [Geometry foundations path](../learning-paths/geometry-foundations.md)
